# Bài học 10: Structured Output (Dữ liệu đầu ra có cấu trúc)

## Vấn đề: Văn bản tự do

Trong các bài học trước, agent trả về **văn bản tự do** — dễ đọc với người, nhưng **rất khó xử lý bằng code**.

Trong Bài học 6, chúng ta đã biết rằng việc chỉ định định dạng đầu ra ngay trong prompt là một kỹ thuật rất mạnh. Ví dụ, bạn có thể viết: "Trả kết quả dưới dạng JSON với các key: title, description." Nhưng cách này phụ thuộc vào việc LLM tuân thủ hướng dẫn một cách hoàn hảo.

**`output_schema`** đưa việc này lên một tầm cao hơn — nó **buộc** agent trả về dữ liệu theo một **định dạng đã định nghĩa sẵn**, đảm bảo 100%. Giống như việc điền vào một biểu mẫu có sẵn thay vì viết một bài luận tự do.

In [ ]:
# Đầu tiên, hãy xem văn bản tự do trông như thế nào
from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude

agent_free = Agent(
    name="Free Text Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=["Create an outline for an SEO article."],
)

response = agent_free.run("Create an outline for an article about 'On-Page SEO Optimization'")
print("--- Văn bản tự do (khó xử lý trong code) ---")
print(response.content)
print(f"\nKiểu dữ liệu: {type(response.content)}")
# Đây chỉ là một chuỗi dài — bạn không thể dễ dàng trích xuất title hay sections!

## Trước hết: JSON là gì?

Trước khi học về structured output, bạn cần hiểu **JSON** — định dạng mà agent sử dụng để trả về dữ liệu có cấu trúc.

**JSON** (JavaScript Object Notation) là một định dạng văn bản phổ biến toàn cầu cho dữ liệu. Nó trông gần như giống hệt dict và list trong Python:

```json
{
  "title": "SEO Guide 2026",
  "word_count": 2000,
  "keywords": ["seo", "ranking", "optimization"],
  "published": true
}
```

**JSON vs Python dict — hãy nhận ra điểm giống nhau:**

| Đặc điểm | Python dict | JSON |
|---------|-------------|------|
| Cặp key-value | `{"name": "Viet"}` | `{"name": "Viet"}` |
| List | `["a", "b"]` | `["a", "b"]` |
| Số | `20` | `20` |
| Boolean | `True` / `False` | `true` / `false` |
| Null | `None` | `null` |

Tại sao JSON quan trọng? Khi agent trả về dữ liệu có cấu trúc, chúng trả về dưới dạng JSON. Python sau đó chuyển đổi nó thành các object bạn có thể sử dụng trong code. Đó chính là nhiệm vụ của `output_schema` — nó bảo agent rằng "hãy trả câu trả lời dưới dạng JSON đúng theo định dạng này."

## Pydantic Models — Tạo "biểu mẫu" cho dữ liệu

**Pydantic** cho phép bạn định nghĩa dữ liệu cần trông như thế nào. Hãy nghĩ nó như việc tạo một **biểu mẫu** cho dữ liệu:

```python
class ArticleOutline(BaseModel):
    title: str          # Trường tiêu đề (văn bản)
    sections: list[str] # Trường danh sách phần (danh sách văn bản)
    target_keyword: str # Trường từ khóa (văn bản)
```

Mỗi trường bao gồm:
- Một **tên** (`title`, `sections`, ...)
- Một **kiểu dữ liệu** (`str` = văn bản, `list[str]` = danh sách văn bản, `int` = số)
- Một **mô tả** (`Field(description="...")`) — giúp agent hiểu cần điền gì vào

In [ ]:
from pydantic import BaseModel, Field

# Định nghĩa "biểu mẫu" cho dàn ý bài viết
class ArticleOutline(BaseModel):
    title: str = Field(description="Article title")
    sections: list[str] = Field(description="List of main sections")
    target_keyword: str = Field(description="Primary keyword")

# Xem cấu trúc của model
print("Cấu trúc ArticleOutline:")
print(ArticleOutline.model_json_schema())

## output_schema — Buộc agent tuân theo biểu mẫu

Khi bạn thêm `output_schema=ArticleOutline` vào agent, nó sẽ:
1. **Đọc biểu mẫu** của bạn (ArticleOutline)
2. **Suy nghĩ** và tạo nội dung
3. **Điền vào biểu mẫu** đúng theo cấu trúc đã định
4. Trả về một **object Python** — bạn có thể truy cập `outline.title`, `outline.sections`, v.v.

```python
agent = Agent(
    output_schema=ArticleOutline,  # Chỉ cần thêm dòng này!
)
```

In [ ]:
# Agent với structured output
agent = Agent(
    name="Outline Creator",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ArticleOutline,
    instructions=["Create an outline for an SEO article."],
)

response = agent.run("Create an outline for an article about 'On-Page SEO Optimization'")
outline = response.content

print(f"Tiêu đề: {outline.title}")
print(f"Từ khóa: {outline.target_keyword}")
print(f"\nCác phần:")
for i, section in enumerate(outline.sections, 1):
    print(f"  {i}. {section}")

## Tại sao Structured Output quan trọng

Dữ liệu có cấu trúc cho phép bạn:

1. **Lưu vào database** — `db.save(outline.title, outline.sections)`
2. **Truyền cho agent khác** — Writer Agent nhận dàn ý từ Researcher Agent
3. **Xuất ra file** — Tạo CSV, JSON, HTML từ dữ liệu
4. **Xử lý trong code** — Đếm số phần, lọc từ khóa, v.v.

Đây chính là nền tảng để xây dựng **pipeline tự động** — dữ liệu được truyền chính xác từ agent này sang agent khác.

In [ ]:
# Truy cập dữ liệu có cấu trúc dễ dàng — giống hệt một object Python bình thường!
print("--- Truy cập dữ liệu có cấu trúc ---")
print(f"Tiêu đề: {outline.title}")
print(f"Từ khóa: {outline.target_keyword}")
print(f"Số lượng phần: {len(outline.sections)}")
print(f"Phần đầu tiên: {outline.sections[0]}")
print(f"Phần cuối cùng: {outline.sections[-1]}")

# Chuyển đổi sang dict hoặc JSON
print(f"\n--- Dạng dictionary ---")
print(outline.model_dump())

print(f"\n--- Dạng JSON ---")
print(outline.model_dump_json(indent=2))

## Tóm tắt Bài học 10

Những gì bạn đã học:
- **Vấn đề** với văn bản tự do: khó xử lý trong code
- **Pydantic BaseModel**: tạo "biểu mẫu" cho dữ liệu (tên, kiểu, mô tả)
- **output_schema**: buộc agent trả về dữ liệu đúng theo định dạng đã chỉ định
- Truy cập dữ liệu dễ dàng: `outline.title`, `outline.sections[0]`
- Chuyển đổi dữ liệu: `.model_dump()` (dict), `.model_dump_json()` (JSON)

**Bài học tiếp theo:** Chúng ta sẽ **nối chuỗi** nhiều agent lại với nhau — đầu ra của agent này trở thành đầu vào của agent kế tiếp!

## Bài tập

Hãy chỉnh sửa schema `ArticleOutline` để thêm một trường mới: `word_count_target` (số nguyên với mô tả "Target word count for the article").

Sau đó tạo một agent với schema đã cập nhật và chạy thử. Kiểm tra xem agent có điền đúng trường mới của bạn không.

Gợi ý:
- Thêm `word_count_target: int = Field(description="...")` vào class
- Truy cập bằng `outline.word_count_target` sau khi chạy

In [ ]:
# Bài tập: Viết code của bạn ở đây
